# Supplier Portal Feature Audit: Performance Hub

**Author:** shazeb.asad | **Snapshot Date:** April 14, 2026

**Source:** [Confluence](https://getyourguide.atlassian.net/wiki/spaces/DA/pages/4205936693)

**Tables used:**
- `production.events.events` — Primary event source; partitioned by `date`. Supplier ID extracted via `CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)`
- `production.supply_analytics.dim_supplier_summary` — Supplier master used to define the relevant supplier denominator and classify managed vs. unmanaged and connected vs. non-connected status
- `production.dwh.dim_tour` — Activity master for relevant activity definition
- `production.dwh.dim_tour_history` — Activity history for first-online date
- `production.dwh.fact_booking` — Booking transactions for L365 booking counts

**Relevant supplier / activity definition (applied to reach queries):**
- Relevant activity: `dim_tour.status = 'active'` AND `dim_tour.gyg_status = 'active'`
- Relevant supplier: `user_status = Active`, has at least one relevant activity, AND (> 1 booking in L365 at supplier level OR any activity first online < 30 days ago)

In [ ]:
SNAPSHOT_DATE        = "2026-04-14"
FULL_ROLLOUT_START   = "2026-03-30"
EARLY_ACCESS_START   = "2026-03-02"
ACTION_RESOLVE_START = "2026-04-03"
BANNER_DISMISS_START = "2026-04-06"

# ─── Relevant Activity ───────────────────────────────────────────
# All three must hold:
#   1. Activity supplier status = Active  (dim_tour.status)
#   2. Activity GYG status      = Active  (dim_tour.gyg_status)
#   3. First went online < 30 days ago  OR  received > 1 booking in L365
#      Online status sourced from dim_tour_history.is_online (MIN = first ever online)
#
# ─── Relevant Supplier ──────────────────────────────────────────
# All of the following must hold:
#   1. Supplier status    = Active      (dim_supplier_summary.user_status)
#   2. > 1 booking in L365 at supplier level  OR  any activity first online < 30 days
#   3. Has at least one relevant activity (as defined above)

RELEVANT_SUPPLIERS_CTE = f"""
activity_bookings_l365 AS (
  SELECT
    tour_id,
    COUNT(*) AS bookings_l365
  FROM production.dwh.fact_booking
  WHERE date_of_checkout >= DATE_SUB(DATE '{SNAPSHOT_DATE}', 365)
    AND is_fraud   = FALSE
    AND status_id IN (1, 2)
  GROUP BY tour_id
),
supplier_bookings_l365 AS (
  SELECT
    supplier_id,
    COUNT(*) AS bookings_l365
  FROM production.dwh.fact_booking
  WHERE date_of_checkout >= DATE_SUB(DATE '{SNAPSHOT_DATE}', 365)
    AND is_fraud   = FALSE
    AND status_id IN (1, 2)
  GROUP BY supplier_id
),
activity_first_online AS (
  -- First date each activity had is_online = TRUE in dim_tour_history
  SELECT
    tour_id,
    CAST(MIN(update_timestamp) AS DATE) AS first_online_date
  FROM production.dwh.dim_tour_history
  WHERE is_online = TRUE
  GROUP BY tour_id
),
relevant_activities AS (
  SELECT DISTINCT
    a.tour_id,
    a.user_id AS supplier_id
  FROM production.dwh.dim_tour a
  LEFT JOIN activity_bookings_l365 ab ON a.tour_id = ab.tour_id
  LEFT JOIN activity_first_online  fo ON a.tour_id = fo.tour_id
  WHERE lower(a.status)     = 'active'
    AND lower(a.gyg_status) = 'active'
    AND (
      fo.first_online_date >= DATE_SUB(DATE '{SNAPSHOT_DATE}', 30)
      OR COALESCE(ab.bookings_l365, 0) > 1
    )
),
activation_period_suppliers AS (
  -- Suppliers with any activity that first went online within the last 30 days
  SELECT DISTINCT a.user_id AS supplier_id
  FROM production.dwh.dim_tour a
  INNER JOIN activity_first_online fo ON a.tour_id = fo.tour_id
  WHERE fo.first_online_date >= DATE_SUB(DATE '{SNAPSHOT_DATE}', 30)
),
relevant_suppliers AS (
  SELECT DISTINCT s.supplier_id
  FROM production.supply_analytics.dim_supplier_summary s
  INNER JOIN relevant_activities         ra  ON s.supplier_id = ra.supplier_id
  LEFT  JOIN supplier_bookings_l365      sb  ON s.supplier_id = sb.supplier_id
  LEFT  JOIN activation_period_suppliers aps ON s.supplier_id = aps.supplier_id
  WHERE lower(s.user_status) = 'active'
    AND (
      COALESCE(sb.bookings_l365, 0) > 1
      OR aps.supplier_id IS NOT NULL
    )
)
"""

---
## Q1: How many relevant suppliers is the Performance Hub reaching?

In [ ]:
# Table 1.1 — Top-line Reach (30 March - 14 April 2026)
df_q1_1 = spark.sql(f"""
WITH
{RELEVANT_SUPPLIERS_CTE},
ph_events AS (
  SELECT
    CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id,
    event_name
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
),
reached AS (
  SELECT DISTINCT supplier_id FROM ph_events
),
page_counts AS (
  SELECT
    COUNT(*)                                                                                                                                               AS total_ph_page_views,
    COUNT(DISTINCT CASE WHEN event_name = 'SupplierPerformancePageRequest'        THEN supplier_id END)                                                  AS unique_bp_suppliers,
    COUNT(DISTINCT CASE WHEN event_name = 'SupplierProductPerformancePageRequest' THEN supplier_id END)                                                  AS unique_pp_suppliers
  FROM ph_events
)
SELECT
  COUNT(DISTINCT rs.supplier_id)                                                                                                                          AS relevant_suppliers,
  COUNT(DISTINCT r.supplier_id)                                                                                                                           AS reached_suppliers,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT r.supplier_id) / COUNT(DISTINCT rs.supplier_id), 1), '%')                                                         AS reach_rate,
  p.total_ph_page_views,
  p.unique_bp_suppliers,
  p.unique_pp_suppliers
FROM relevant_suppliers rs
LEFT JOIN reached r ON rs.supplier_id = r.supplier_id
CROSS JOIN page_counts p
GROUP BY p.total_ph_page_views, p.unique_bp_suppliers, p.unique_pp_suppliers
""")

display(df_q1_1)

In [ ]:
# Table 1.2 — Reach by Managed vs Unmanaged
df_q1_2 = spark.sql(f"""
WITH
{RELEVANT_SUPPLIERS_CTE},
reached AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
)
SELECT
  s.is_managed                                                                                                                                             AS segment,
  COUNT(DISTINCT rs.supplier_id)                                                                                                                          AS relevant_suppliers,
  COUNT(DISTINCT CASE WHEN r.supplier_id IS NOT NULL THEN rs.supplier_id END)                                                                            AS reached,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT CASE WHEN r.supplier_id IS NOT NULL THEN rs.supplier_id END) / COUNT(DISTINCT rs.supplier_id), 1), '%')            AS reach_rate
FROM relevant_suppliers rs
JOIN  production.supply_analytics.dim_supplier_summary s ON rs.supplier_id = s.supplier_id
LEFT JOIN reached r ON rs.supplier_id = r.supplier_id
GROUP BY s.is_managed
ORDER BY reach_rate DESC
""")

display(df_q1_2)

In [ ]:
# Table 1.2b — Reach by Connected vs Non-Connected
df_q1_2b = spark.sql(f"""
WITH
{RELEVANT_SUPPLIERS_CTE},
reached AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
)
SELECT
  s.is_connected                                                                                                                                           AS segment,
  COUNT(DISTINCT rs.supplier_id)                                                                                                                          AS relevant_suppliers,
  COUNT(DISTINCT CASE WHEN r.supplier_id IS NOT NULL THEN rs.supplier_id END)                                                                            AS reached,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT CASE WHEN r.supplier_id IS NOT NULL THEN rs.supplier_id END) / COUNT(DISTINCT rs.supplier_id), 1), '%')            AS reach_rate
FROM relevant_suppliers rs
JOIN  production.supply_analytics.dim_supplier_summary s ON rs.supplier_id = s.supplier_id
LEFT JOIN reached r ON rs.supplier_id = r.supplier_id
GROUP BY s.is_connected
ORDER BY reach_rate DESC
""")

display(df_q1_2b)

---
## Q2: How active are suppliers with the Performance Hub?

In [ ]:
# Table 2.1 — Monthly Active Suppliers (MAU)
df_q2_1 = spark.sql(f"""
SELECT
  CASE
    WHEN date BETWEEN '{EARLY_ACCESS_START}' AND '2026-03-31' THEN 'March 2026 (early access)'
    WHEN date BETWEEN '2026-04-01' AND '{SNAPSHOT_DATE}'      THEN 'April 2026 (1-14)'
  END                                                                                                          AS month,
  COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT))                               AS mau_bp,
  COUNT(*)                                                                                                     AS bp_page_views,
  MIN(date)                                                                                                    AS period_start
FROM production.events.events
WHERE date BETWEEN '{EARLY_ACCESS_START}' AND '{SNAPSHOT_DATE}'
  AND event_name = 'SupplierPerformancePageRequest'
GROUP BY
  CASE
    WHEN date BETWEEN '{EARLY_ACCESS_START}' AND '2026-03-31' THEN 'March 2026 (early access)'
    WHEN date BETWEEN '2026-04-01' AND '{SNAPSHOT_DATE}'      THEN 'April 2026 (1-14)'
  END
ORDER BY period_start
""")

display(df_q2_1)

In [ ]:
# Table 2.2 — Weekly Active Suppliers (WAU) Post Full Rollout
df_q2_2 = spark.sql(f"""
WITH ph_events AS (
  SELECT
    CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id,
    event_name,
    date
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN (
      'SupplierPerformancePageRequest',
      'SupplierBusinessPerformanceDateRangeChange',
      'SupplierProductPerformancePageRequest',
      'SupplierPerformanceTopActionsVisible',
      'SupplierProductPerformanceGraphChange',
      'SupplierOptimizeProductsWidgetViewDetailsClick',
      'SupplierPerformanceBannerDismiss',
      'SupplierProductPerformanceActionResolve',
      'SupplierProductPerformanceActionDismiss'
    )
),
week_buckets AS (
  SELECT
    *,
    CASE
      WHEN date BETWEEN '2026-03-30' AND '2026-04-05' THEN '30 Mar - 5 Apr'
      WHEN date BETWEEN '2026-04-06' AND '2026-04-12' THEN '6 Apr - 12 Apr'
      WHEN date BETWEEN '2026-04-13' AND '2026-04-14' THEN '13-14 Apr (partial)'
    END AS week,
    CASE
      WHEN date BETWEEN '2026-03-30' AND '2026-04-05' THEN 1
      WHEN date BETWEEN '2026-04-06' AND '2026-04-12' THEN 2
      WHEN date BETWEEN '2026-04-13' AND '2026-04-14' THEN 3
    END AS week_order
  FROM ph_events
)
SELECT
  week,
  COUNT(DISTINCT supplier_id)                                                                                                                              AS wau_all_ph,
  COUNT(DISTINCT CASE WHEN event_name = 'SupplierPerformancePageRequest'        THEN supplier_id END)                                                    AS wau_bp,
  COUNT(DISTINCT CASE WHEN event_name = 'SupplierProductPerformancePageRequest' THEN supplier_id END)                                                    AS wau_pp,
  COUNT(*)                                                                                                                                                 AS total_events,
  week_order
FROM week_buckets
GROUP BY week, week_order
ORDER BY week_order
""")

display(df_q2_2)

---
## Q3: How are suppliers using each section of the Performance Hub?

In [ ]:
# Table 3.1 — Usage Summary by Metric View (30 March - 14 April 2026)
df_q3_1 = spark.sql(f"""
WITH bp_events AS (
  SELECT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierPerformancePageRequest'
),
pp_events AS (
  SELECT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformancePageRequest'
),
all_pv AS (
  SELECT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
)
SELECT
  metric_view,
  unique_suppliers,
  page_views,
  ROUND(CAST(page_views AS DOUBLE) / unique_suppliers, 1) AS avg_pv_per_visitor
FROM (
  SELECT 'Overall (BP + PP)'    AS metric_view, COUNT(DISTINCT supplier_id) AS unique_suppliers, COUNT(*) AS page_views FROM all_pv
  UNION ALL
  SELECT 'Business Performance',                COUNT(DISTINCT supplier_id),                     COUNT(*) FROM bp_events
  UNION ALL
  SELECT 'Product Performance',                 COUNT(DISTINCT supplier_id),                     COUNT(*) FROM pp_events
) t
ORDER BY
  CASE metric_view
    WHEN 'Overall (BP + PP)'    THEN 1
    WHEN 'Business Performance' THEN 2
    WHEN 'Product Performance'  THEN 3
  END
""")

display(df_q3_1)

In [ ]:
# Table 3.2 — Business Performance Page: Engagement Funnel (30 March - 14 April 2026)
df_q3_2 = spark.sql(f"""
WITH bp_visitors AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierPerformancePageRequest'
),
date_range_changers AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierBusinessPerformanceDateRangeChange'
),
ra_widget_viewers AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierPerformanceTopActionsVisible'
),
ra_widget_clickers AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierOptimizeProductsWidgetViewDetailsClick'
)
SELECT
  COUNT(DISTINCT bv.supplier_id)                                                                                                                          AS bp_visitors,
  COUNT(DISTINCT dr.supplier_id)                                                                                                                          AS date_range_changers,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT dr.supplier_id) / COUNT(DISTINCT bv.supplier_id), 1), '%')                                                        AS date_range_change_pct,
  COUNT(DISTINCT rv.supplier_id)                                                                                                                          AS ra_widget_visible,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT rv.supplier_id) / COUNT(DISTINCT bv.supplier_id), 1), '%')                                                        AS ra_widget_visible_pct,
  COUNT(DISTINCT rc.supplier_id)                                                                                                                          AS ra_widget_clicked,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT rc.supplier_id) / COUNT(DISTINCT bv.supplier_id), 1), '%')                                                        AS ra_widget_click_pct
FROM bp_visitors bv
LEFT JOIN date_range_changers dr ON bv.supplier_id = dr.supplier_id
LEFT JOIN ra_widget_viewers   rv ON bv.supplier_id = rv.supplier_id
LEFT JOIN ra_widget_clickers  rc ON bv.supplier_id = rc.supplier_id
""")

display(df_q3_2)

In [ ]:
# Table 3.3 — Product Performance Pages: Engagement Funnel (30 March - 14 April 2026)
# Action resolve from PP only available from 2026-04-03
df_q3_3 = spark.sql(f"""
WITH pp_visitors AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformancePageRequest'
),
graph_changers AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformanceGraphChange'
),
action_resolvers AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{ACTION_RESOLVE_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformanceActionResolve'
)
SELECT
  COUNT(DISTINCT ppv.supplier_id)                                                                                                                         AS pp_visitors,
  COUNT(DISTINCT gc.supplier_id)                                                                                                                          AS graph_changers,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT gc.supplier_id) / COUNT(DISTINCT ppv.supplier_id), 1), '%')                                                       AS graph_change_pct,
  COUNT(DISTINCT ar.supplier_id)                                                                                                                          AS action_resolvers,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT ar.supplier_id) / COUNT(DISTINCT ppv.supplier_id), 1), '%')                                                       AS action_resolve_pct
FROM pp_visitors ppv
LEFT JOIN graph_changers   gc ON ppv.supplier_id = gc.supplier_id
LEFT JOIN action_resolvers ar ON ppv.supplier_id = ar.supplier_id
""")

display(df_q3_3)

---
## Q4: How deeply do suppliers engage per visit?

In [ ]:
# Table 4.1 — Per-User Engagement Depth (2 March - 14 April 2026)
df_q4_1 = spark.sql(f"""
WITH supplier_events AS (
  SELECT
    CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id,
    event_name
  FROM production.events.events
  WHERE date BETWEEN '{EARLY_ACCESS_START}' AND '{SNAPSHOT_DATE}'
    AND event_name IN (
      'SupplierPerformancePageRequest',
      'SupplierBusinessPerformanceDateRangeChange',
      'SupplierProductPerformancePageRequest',
      'SupplierPerformanceTopActionsVisible',
      'SupplierProductPerformanceGraphChange',
      'SupplierOptimizeProductsWidgetViewDetailsClick',
      'SupplierPerformanceBannerDismiss',
      'SupplierProductPerformanceActionResolve',
      'SupplierProductPerformanceActionDismiss'
    )
),
per_supplier AS (
  SELECT
    supplier_id,
    COUNT(*)                                                                                                                                              AS total_events,
    SUM(CASE WHEN event_name = 'SupplierPerformancePageRequest'        THEN 1 ELSE 0 END)                                                               AS bp_page_views,
    SUM(CASE WHEN event_name = 'SupplierProductPerformancePageRequest' THEN 1 ELSE 0 END)                                                               AS pp_page_views,
    SUM(CASE WHEN event_name NOT IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest') THEN 1 ELSE 0 END)                      AS non_pv_interactions
  FROM supplier_events
  GROUP BY supplier_id
)
SELECT 'Business Performance page views per visitor'   AS metric, ROUND(AVG(bp_page_views),       1) AS mean_value, ROUND(PERCENTILE(bp_page_views,       0.5), 1) AS median_value FROM per_supplier WHERE bp_page_views > 0
UNION ALL
SELECT 'Product Performance page views per visitor',              ROUND(AVG(pp_page_views),       1),              ROUND(PERCENTILE(pp_page_views,       0.5), 1) FROM per_supplier WHERE pp_page_views > 0
UNION ALL
SELECT 'Non-pageview interactions per visitor',                   ROUND(AVG(non_pv_interactions), 1),              ROUND(PERCENTILE(non_pv_interactions, 0.5), 1) FROM per_supplier
UNION ALL
SELECT 'Total events per visitor',                                ROUND(AVG(total_events),        1),              ROUND(PERCENTILE(total_events,        0.5), 1) FROM per_supplier
""")

display(df_q4_1)

---
## Q5: What functionality is most used?

In [ ]:
# Table 5.1 — Event Volume Breakdown (2 March - 14 April 2026)
df_q5_1 = spark.sql(f"""
SELECT
  event_name,
  COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT))  AS unique_suppliers,
  COUNT(*)                                                                        AS total_events
FROM production.events.events
WHERE date BETWEEN '{EARLY_ACCESS_START}' AND '{SNAPSHOT_DATE}'
  AND event_name IN (
    'SupplierPerformancePageRequest',
    'SupplierBusinessPerformanceDateRangeChange',
    'SupplierProductPerformancePageRequest',
    'SupplierPerformanceTopActionsVisible',
    'SupplierProductPerformanceGraphChange',
    'SupplierOptimizeProductsWidgetViewDetailsClick',
    'SupplierPerformanceBannerDismiss',
    'SupplierProductPerformanceActionResolve',
    'SupplierProductPerformanceActionDismiss',
    'SupplierProductPerformanceActionFilter'
  )
GROUP BY event_name
ORDER BY total_events DESC
""")

display(df_q5_1)

---
## Q6: Are suppliers taking action from the Performance Hub?

In [ ]:
# Table 6.1 — Action-Taking From Performance Hub (30 March - 14 April 2026)
# Action resolve/dismiss events started tracking from 2026-04-03
# Banner dismiss started tracking from 2026-04-06
df_q6_1 = spark.sql(f"""
WITH pp_visitors AS (
  SELECT COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)) AS cnt
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformancePageRequest'
),
pp_resolvers AS (
  SELECT
    COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)) AS unique_cnt,
    COUNT(*) AS total_cnt
  FROM production.events.events
  WHERE date BETWEEN '{ACTION_RESOLVE_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformanceActionResolve'
),
pp_dismissers AS (
  SELECT
    COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)) AS unique_cnt,
    COUNT(*) AS total_cnt
  FROM production.events.events
  WHERE date BETWEEN '{ACTION_RESOLVE_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierProductPerformanceActionDismiss'
),
bp_visitors AS (
  SELECT COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)) AS cnt
  FROM production.events.events
  WHERE date BETWEEN '{FULL_ROLLOUT_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierPerformancePageRequest'
),
banner_dismissers AS (
  SELECT COUNT(DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT)) AS unique_cnt
  FROM production.events.events
  WHERE date BETWEEN '{BANNER_DISMISS_START}' AND '{SNAPSHOT_DATE}'
    AND event_name = 'SupplierPerformanceBannerDismiss'
)
SELECT
  ppv.cnt                                                                                                                                                  AS pp_visitors,
  ppr.unique_cnt                                                                                                                                           AS pp_resolvers,
  CONCAT(ROUND(100.0 * ppr.unique_cnt / ppv.cnt, 1), '%')                                                                                               AS resolve_rate_pct_of_pp_visitors,
  ppr.total_cnt                                                                                                                                            AS total_actions_resolved_via_pp,
  ppd.unique_cnt                                                                                                                                           AS pp_dismissers,
  CONCAT(ROUND(100.0 * ppd.unique_cnt / ppv.cnt, 1), '%')                                                                                               AS dismiss_rate_pct_of_pp_visitors,
  ppd.total_cnt                                                                                                                                            AS total_actions_dismissed_via_pp,
  bpv.cnt                                                                                                                                                  AS bp_visitors,
  bd.unique_cnt                                                                                                                                            AS banner_dismissers,
  CONCAT(ROUND(100.0 * bd.unique_cnt / bpv.cnt, 1), '%')                                                                                               AS banner_dismiss_rate_pct_of_bp_visitors
FROM pp_visitors      ppv
CROSS JOIN pp_resolvers   ppr
CROSS JOIN pp_dismissers  ppd
CROSS JOIN bp_visitors    bpv
CROSS JOIN banner_dismissers bd
""")

display(df_q6_1)

---
## Q7: Are suppliers returning to the Performance Hub?

In [ ]:
# Table 7.1 — Early Cohort Returning Rate
# Cohort: suppliers who visited any PH page between EARLY_ACCESS_START and 2026-03-31
# Returned: same suppliers with any PH page view between 2026-04-01 and SNAPSHOT_DATE
df_q7_1 = spark.sql(f"""
WITH march_cohort AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '{EARLY_ACCESS_START}' AND '2026-03-31'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
),
april_visitors AS (
  SELECT DISTINCT CAST(get_json_object(json_event, '$.supplier_id') AS BIGINT) AS supplier_id
  FROM production.events.events
  WHERE date BETWEEN '2026-04-01' AND '{SNAPSHOT_DATE}'
    AND event_name IN ('SupplierPerformancePageRequest', 'SupplierProductPerformancePageRequest')
)
SELECT
  'March early access (Mar 2-31)'                                                                               AS cohort,
  COUNT(DISTINCT mc.supplier_id)                                                                                AS month_1_visitors,
  COUNT(DISTINCT av.supplier_id)                                                                                AS returned_in_month_2,
  CONCAT(ROUND(100.0 * COUNT(DISTINCT av.supplier_id) / COUNT(DISTINCT mc.supplier_id), 1), '%')               AS returning_rate
FROM march_cohort mc
LEFT JOIN april_visitors av ON mc.supplier_id = av.supplier_id
""")

display(df_q7_1)

---
## Appendix: Key Metrics Glossary

| Term | Definition |
|------|-----------|
| **Relevant activity** | Activity where: `dim_tour.status = 'active'` AND `dim_tour.gyg_status = 'active'` |
| **Relevant supplier** | Supplier where: `user_status = Active`, has at least one relevant activity, AND (> 1 booking in L365 at supplier level OR any activity first online < 30 days) |
| **Activation period** | A 30-day window beginning when an activity first goes online. Activities in this period are included in the relevant supplier definition even with fewer than 2 bookings in L365 |
| **Snapshot date** | 2026-04-14 — upper bound of all analysis windows |
| **Full rollout start** | 2026-03-30 — date when the Performance Hub homepage widget was deployed to all suppliers. Primary analysis window lower bound |
| **Early access start** | 2026-03-02 — first date Performance Hub events were recorded. Used for full availability window (engagement depth, event volume, cohort retention) |
| **Business Performance page** | Main `/performance` page of the Supplier Portal. Tracked via `SupplierPerformancePageRequest` |
| **Product Performance pages** | Per-product sub-pages under the Performance Hub. Tracked via `SupplierProductPerformancePageRequest` |
| **Reached supplier** | A relevant supplier who visited at least one PH page in the post-rollout window (30 Mar - 14 Apr 2026) |
| **Reach rate** | Share of relevant suppliers who visited at least one PH page in the post-rollout window |
| **MAU** | Unique suppliers with at least one BP page view in a calendar month |
| **WAU** | Unique suppliers with at least one PH event in a calendar week |
| **Engagement depth** | Per-visitor count of non-pageview interactions. Measures active use beyond mere page access |
| **Returning supplier rate** | Share of month N visitors who return in month N+1. Primary stickiness metric |
| **Partition filter** | Always filter `date BETWEEN '...' AND '...'` to avoid full table scans on partitioned `production.events.events` |